In [0]:
"""setup cell containing all imports and constants"""
import os, sys, json, time
from datetime import date, timedelta

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))

from nba_pipeline.api_client import fetch_scoreboard_json, fetch_game_summary_json
from nba_pipeline.storage import land_raw

RAW_BASE = "/Volumes/workspace/nba/raw"
print("ready")

In [0]:
import os, sys
from datetime import date

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))

from nba_pipeline.api_client import fetch_scoreboard_json, fetch_game_summary_json

sb = fetch_scoreboard_json(date(2025, 12, 25))
print("scoreboard chars:", len(sb))

gs = fetch_game_summary_json("401809238")
print("summary chars:", len(gs))

In [0]:
from nba_pipeline.storage import land_raw

RAW_BASE = "/Volumes/workspace/nba/raw"

p1 = land_raw(sb, RAW_BASE, "scoreboard", "game_date=2025-12-25")
p2 = land_raw(gs, RAW_BASE, "game_summary", "game_id=401809238")
print(p1)
print(p2)
print(os.listdir(f"{RAW_BASE}/scoreboard"))

In [0]:
import json
import time
from datetime import date, timedelta

START = date(2026, 10, 20)   # opening night
END = date(2026, 11, 15)     # yesterday

results = {"ok": 0, "failed": []}

d = START
while d <= END:
    try:
        payload = fetch_scoreboard_json(d)
        land_raw(payload, RAW_BASE, "scoreboard", f"game_date={d.isoformat()}")
        n_games = len(json.loads(payload).get("events", []))
        print(f"{d}: {n_games} games")
        results["ok"] += 1
    except Exception as exc:
        print(f"{d}: FAILED: {exc}")
        results["failed"].append(d.isoformat())
    time.sleep(1.5)   # be polite to the API
    d += timedelta(days=1)

print(results)
print(len(os.listdir(f"{RAW_BASE}/scoreboard")), "files in the volume")

In [0]:
import glob
from collections import Counter

games = []
for path in sorted(glob.glob(f"{RAW_BASE}/scoreboard/game_date=*.json")):
    game_date = os.path.basename(path).split("=")[1].replace(".json", "")
    if game_date < "2026-10-20":
        continue   # skip the Dec 2025 test file
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    for ev in data.get("events", []):
        games.append({
            "game_date": game_date,
            "game_id": ev["id"],
            "name": ev["name"],
            "status": ev["status"]["type"]["name"],
        })

print(len(games), "games")
print(Counter(g["status"] for g in games))
print(games[0])

In [0]:
"""Removing test files"""
test_files = [
    f"{RAW_BASE}/scoreboard/game_date=2025-12-25.json",
    f"{RAW_BASE}/game_summary/game_id=401809238.json",
]

for p in test_files:
    if os.path.exists(p):
        os.remove(p)
        print("deleted", p)
    else:
        print("not found", p)

print(len(os.listdir(f"{RAW_BASE}/scoreboard")), "scoreboard files")
print(os.listdir(f"{RAW_BASE}/game_summary"))

In [0]:
# --- Fetch the raw game summary (box score) for every game in `games` ---
# Relies on: `games` (from the previous step), RAW_BASE, and the setup-cell imports.

ok, skipped, failed = 0, 0, []

for i, g in enumerate(games, start=1):
    game_id = g["game_id"]
    out_path = f"{RAW_BASE}/game_summary/game_id={game_id}.json"

    # Resume support: finished games never change, so if the file is already
    # in the volume we skip the API call. This lets us re-run safely after
    # an interruption without re-fetching everything.
    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        # Fetch the raw JSON text (retries and backoff happen inside the client)
        payload = fetch_game_summary_json(game_id)
        # Save it untouched to the volume; file name is built from the game ID
        land_raw(payload, RAW_BASE, "game_summary", f"game_id={game_id}")
        ok += 1
    except Exception as exc:
        # One bad game shouldn't stop the whole run; record it and move on
        print(f"{game_id} ({g['name']}): FAILED: {exc}")
        failed.append(game_id)

    # Print a progress line every 20 games so the output stays readable
    if i % 20 == 0:
        print(f"progress: {i}/{len(games)} (ok={ok}, skipped={skipped}, failed={len(failed)})")

    # Throttle: pause between requests to stay polite to the API
    time.sleep(1.5)

# Final summary: counts plus the IDs of any games that need a retry
print({"ok": ok, "skipped": skipped, "failed": failed})
print(len(os.listdir(f"{RAW_BASE}/game_summary")), "files in game_summary")

In [0]:
# --- Read the raw game summary files into a Spark DataFrame ---
from pyspark.sql import functions as F   # F gives us Spark's built-in column functions

# 1. Describe the read. The "text" format reads files as plain text.
#    wholetext=True makes Spark treat each FILE as ONE row (instead of one row per line).
#    NOTHING is read yet: this just builds a plan (lazy evaluation).
raw_df = (
    spark.read
    .format("text")
    .option("wholetext", True)
    .load(f"{RAW_BASE}/game_summary/")
)

# 2. Shape it into our Bronze layout. Still lazy: we're only adding to the plan.
#    - raw_json:    the file's full contents, untouched
#    - source_file: which file each row came from (_metadata is a hidden column
#                   Databricks provides on file-based reads)
#    - ingested_at: a timestamp of when we loaded it (audit trail)
bronze_df = raw_df.select(
    F.col("value").alias("raw_json"),
    F.col("_metadata.file_path").alias("source_file"),
    F.current_timestamp().alias("ingested_at"),
)

# 3. Inspect the plan's structure. printSchema is cheap: it doesn't read the data.
bronze_df.printSchema()

# 4. Now trigger actions. These make Spark actually read the files.
print(bronze_df.count(), "rows")   # action: count rows

# Show the file name and the JSON size per row (length() is another column function).
bronze_df.select("source_file", F.length("raw_json").alias("json_chars")).show(5, truncate=80)

In [0]:
# --- Write the Bronze DataFrame to a Delta table ---

TABLE = "workspace.nba.bronze_game_summary"   # catalog.schema.table

# This is the ACTION: Spark now actually reads the 197 files and writes the table.
(
    bronze_df.write
    .format("delta")          # store as Delta (Parquet files + transaction log)
    .mode("overwrite")        # replace the table if it already exists (re-runnable)
    .saveAsTable(TABLE)       # register it in Unity Catalog under this name
)

# Verify: read the table back by name and count its rows.
print(spark.table(TABLE).count(), "rows in", TABLE)

# Show the table's transaction log. Each write creates a new version here.
display(spark.sql(f"DESCRIBE HISTORY {TABLE}"))

In [0]:
# --- Reusable Bronze loader: raw files in a volume folder -> Delta table ---
from pyspark.sql import functions as F

def load_bronze(entity: str, table: str) -> int:
    """Read every raw file in RAW_BASE/<entity>/ (one row per file) and
    write it to a Delta table. Returns the table's row count."""

    # Describe the read (lazy): treat each file as one row of text
    raw_df = (
        spark.read
        .format("text")
        .option("wholetext", True)
        .load(f"{RAW_BASE}/{entity}/")
    )

    # Add the Bronze metadata columns (still lazy)
    bronze_df = raw_df.select(
        F.col("value").alias("raw_json"),
        F.col("_metadata.file_path").alias("source_file"),
        F.current_timestamp().alias("ingested_at"),
    )

    # The ACTION: read the files and write the Delta table (overwrite = re-runnable)
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(table)

    # Read the table back and return its row count as a sanity check
    return spark.table(table).count()


# Call it for the scoreboard files
n = load_bronze("scoreboard", "workspace.nba.bronze_scoreboard")
print(n, "rows in workspace.nba.bronze_scoreboard")

# Peek at the table: file name and JSON size per row
spark.table("workspace.nba.bronze_scoreboard") \
    .select("source_file", F.length("raw_json").alias("json_chars")) \
    .show(3, truncate=80)